# Lab 08 · Content Understanding Alternative (Notebook Walkthrough)

**Concept.** The `content_understanding` profile switches the Search-managed extractor to the Azure Content Understanding skill, which performs semantic chunking and richer structure-aware extraction inside the skillset itself. This is an optional, advanced comparison against the earlier `DocumentExtractionSkill` path.

> This lab binds to the billable Foundry resource via the skillset's managed identity. If your environment is not enabled for it, the ingest cell will report the failure and you can skip to the comparison notebook.

## Step 1 — Bootstrap

In [1]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR if (NB_DIR / 'lab_runtime.py').exists() else NB_DIR / 'notebooks'))
import lab_runtime as lab

info = lab.bootstrap()
info

{'repo_root': 'C:\\Users\\samsonlee\\rag-on-azure-lab',
 'env_values_loaded': 89,
 'search_endpoint': 'https://aisearchlabl4qxbh.search.windows.net',
 'search_configured': True,
 'embedding_deployment': 'text-embedding-3-large-vector',
 'chat_deployment': 'gpt-5-4-mini-chat',
 'agentic_planning_model': 'gpt-5-4-mini-search',
 'canonical_index': 'ai-search-lab-index'}

## Step 2 — Ingest with Content Understanding

In [2]:
try:
    job = lab.ingest(skill_profile='content_understanding', reuse=True)
    print(lab.chunk_overview(job))
except Exception as exc:
    job = None
    print('Content Understanding ingestion not available in this environment:')
    print(' ', exc)

Reusing existing 'content_understanding' ingestion (doc_id=1380815d, 414 chunks). Pass reuse=False to force a fresh run.
{'doc_id': '1380815dfc27477b86f2257e3e6456e1', 'skill_profile': 'content_understanding', 'chunk_count': 414, 'avg_tokens': 208.4, 'max_tokens': 1976, 'chunks_with_summary': 414, 'chunks_with_keyword_hints': 414, 'chunks_with_image_description': 0}


## Step 3 — Compare chunk boundaries against baseline

Content Understanding tends to produce semantically coherent chunks; compare the token distribution with the baseline profile.

In [3]:
import pandas as pd

if job is not None:
    base = lab.find_existing_job(skill_profile_id='baseline_extract', file_name='Deep Excavation Design and Construction.pdf')
    rows = []
    for label, j in [('baseline_extract', base), ('content_understanding', job)]:
        if j is None:
            continue
        ov = lab.chunk_overview(j)
        rows.append({'profile': label, 'chunks': ov['chunk_count'], 'avg_tokens': ov['avg_tokens'], 'max_tokens': ov['max_tokens']})
    display(pd.DataFrame(rows))
else:
    print('Skipped — no Content Understanding job.')

,profile,chunks,avg_tokens,max_tokens
0,baseline_extract,412,200.4,777
1,content_understanding,414,208.4,1976


## Step 4 — Query the corpus

In [4]:
Q2 = 'Given a site with high groundwater and clay layers, what are the key excavation risks and design considerations?'

if job is not None:
    resp = lab.ask(Q2, job=job, retrieval_mode='hybrid', record_as='lab08_contentunderstanding_hybrid')
    lab.show_answer(resp, max_citations=4)
else:
    print('Skipped — no Content Understanding job.')

[retrieval_mode=hybrid | scoring_profile=default | citations=8]

Loss of overall stability is likely to occur in excavations near a steeply-sloping site with a high groundwater table, or where a weak subsoil layer (e.g. loose sand or soft clay) is present below the embedded wall. [5]

Supporting evidence:
- In order to minimise disturbance to the adjacent ground due to the boring operation, the pressure of the compressed air should be carefully assessed and monitored, especially for sites with a high groundwater table and thick layers of loose fill, bouldery colluvium or rockfill. Trial boring is usually conducted and should be aimed to determine site-specific  [3]
- As discussed in Section 2.1.2, the differential piezometric pressures between marine clay and alluvial sand layers could lead to porewater pressure dissipation and associated consolidation settlement in the clay layer even when the groundwater level in an overlying fill layer remains stable. [6]

Citations:
  [5] 7.2 Overa

## Takeaways

- Content Understanding moves semantic chunking + structure extraction **into the skillset**.
- Compare its chunk boundaries and retrieval quality with the built-in `DocumentExtractionSkill` profiles.

Next: the **comparison notebook** lays every method side by side.